# Práctica 1: Exploración de LLMs con API y LangChain

## Objetivos
- Aplicar los conceptos de conexión API directa y con LangChain
- Experimentar con diferentes parámetros (temperature, max_tokens, modelos)
- Implementar streaming y memoria conversacional
- Resolver ejercicios prácticos de forma autónoma

## Instrucciones
Completa los ejercicios en orden. Cada sección tiene celdas de código para que implementes las soluciones.

## Configuración Inicial

In [1]:
import os

# Carga de credenciales: funciona igual en Google Colab y en local (.env)
try:
    from google.colab import userdata  # type: ignore
    os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
except ImportError:
    from dotenv import load_dotenv
    load_dotenv()

assert os.getenv("GROQ_API_KEY"), "Falta GROQ_API_KEY (Colab: Secrets · local: archivo .env)"

MODELO = os.getenv("GROQ_MODEL", "llama-3.3-70b-versatile")
MODELO_RAPIDO = os.getenv("GROQ_MODEL_FAST", "llama-3.1-8b-instant")

from groq import Groq

client = Groq()  # lee GROQ_API_KEY del entorno
print("✅ Cliente configurado correctamente")

✅ Cliente configurado correctamente


---
## Ejercicio 1: Comparación de Modelos

**Instrucción:** Prueba al menos 3 modelos diferentes de Groq (`llama-3.1-8b-instant`,
`llama-3.3-70b-versatile`, `openai/gpt-oss-20b`) con el mismo prompt.
Observa las diferencias en velocidad, calidad y cantidad de tokens usados.

> El catálogo vigente está en [console.groq.com/docs/models](https://console.groq.com/docs/models).
> Si alguno de esos identificadores ya no existe, reemplázalo por otro del catálogo.
> Recuerda que la capa gratuita permite ~30 peticiones por minuto: no ejecutes esta celda en bucle.

In [2]:
# Escribe aquí tu código para probar diferentes modelos
modelos = ["llama-3.1-8b-instant", "llama-3.3-70b-versatile", "openai/gpt-oss-20b"]
prompt = "Explica la diferencia entre IA débil y IA fuerte en 3 oraciones."

for modelo in modelos:
    print(f"\n{'='*50}")
    print(f"Modelo: {modelo}")
    print('='*50)
    response = client.chat.completions.create(
        model=modelo,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.3,
        max_tokens=200
    )
    print(f"Respuesta: {response.choices[0].message.content}")
    print(f"Tokens: {response.usage.total_tokens}")


Modelo: llama-3.1-8b-instant


Respuesta: La IA débil se enfoca en resolver problemas específicos y realizar tareas concretas, como reconocimiento de voz, traducción de texto o recomendación de productos, utilizando algoritmos y técnicas de aprendizaje automático. Por otro lado, la IA fuerte busca crear sistemas que puedan pensar y razonar de manera similar a la inteligencia humana, con la capacidad de aprender, razonar y tomar decisiones de manera autónoma y generalizable.

En resumen, la IA débil es especializada y se enfoca en tareas específicas, mientras que la IA fuerte es más general y busca crear sistemas que puedan abordar problemas complejos y multifacéticos de manera autónoma y creativa.
Tokens: 212

Modelo: llama-3.3-70b-versatile


Respuesta: La IA débil se refiere a sistemas de inteligencia artificial diseñados para realizar tareas específicas y limitadas, como reconocimiento de patrones o procesamiento de lenguaje natural, sin tener una comprensión profunda o conciencia de su entorno. Por otro lado, la IA fuerte se refiere a sistemas que poseen una inteligencia generalizada y comparable a la humana, capaces de aprender, razonar y tomar decisiones de manera autónoma y flexible. La principal diferencia entre ambas es que la IA débil se enfoca en resolver problemas específicos, mientras que la IA fuerte busca replicar la inteligencia humana en su totalidad.
Tokens: 199

Modelo: openai/gpt-oss-20b


Respuesta: La IA débil (o estrecha) está diseñada para realizar tareas específicas, como reconocimiento de voz o juegos, sin comprender ni sentir; opera dentro de límites predefinidos y no posee conciencia. La IA fuerte, en cambio, aspira a replicar la inteligencia humana completa, incluyendo la capacidad de razonamiento, aprendizaje autónomo y autoconciencia, lo que le permitiría entender y responder a cualquier situación como un ser humano. En resumen, la IA débil es funcional pero limitada a tareas concretas, mientras que la IA fuerte busca una inteligencia general y consciente.
Tokens: 243


---
## Ejercicio 2: Efecto de la Temperatura

**Instrucción:** Usa el mismo modelo y prompt con diferentes valores de temperature (0.0, 0.5, 1.0, 1.5). 
Ejecuta cada uno 3 veces y analiza la variabilidad de las respuestas.

In [3]:
# Escribe aquí tu código para experimentar con temperature
temperaturas = [0.0, 0.5, 1.0, 1.5]
prompt = "Inventa un nombre creativo para un asistente de IA educativo."

for temp in temperaturas:
    print(f"\n--- Temperature = {temp} ---")
    for i in range(3):
        response = client.chat.completions.create(
            model=MODELO_RAPIDO,
            messages=[{"role": "user", "content": prompt}],
            temperature=temp,
            max_tokens=50
        )
        print(f"  Intento {i+1}: {response.choices[0].message.content}")


--- Temperature = 0.0 ---


  Intento 1: Aquí te presento algunas opciones creativas para un asistente de IA educativo:

1. **Lumin**: un nombre que sugiere iluminación y conocimiento.
2. **Cerebro**: un nombre que evoca la


  Intento 2: Aquí te presento algunas opciones creativas para un asistente de IA educativo:

1. **Lumin**: un nombre que sugiere iluminación y conocimiento.
2. **Cerebro**: un nombre que evoca la


  Intento 3: Aquí te presento algunas opciones creativas para un asistente de IA educativo:

1. **Lumin**: un nombre que sugiere iluminación y conocimiento.
2. **Cerebro**: un nombre que evoca la

--- Temperature = 0.5 ---


  Intento 1: Algunas opciones creativas para un asistente de IA educativo podrían ser:

1. **Lumin**: un nombre que sugiere iluminación y conocimiento.
2. **Cerebro**: un nombre que evoca la


...

  Intento 2: Aquí te presento algunas ideas creativas para un nombre de asistente de IA educativo:

1. **Cerebro**: un nombre sencillo y reconocible que sugiere un intelecto agudo y una gran capacidad para


  Intento 3: Aquí te propongo algunos nombres creativos para un asistente de IA educativo:

1. **Lumin**: Un nombre que evoca la idea de iluminación y conocimiento.
2. **Cerebro**: Un nombre

--- Temperature = 1.5 ---


  Intento 1: Aquí te presento algunas sugerencias de nombres creativos para un asistente de IA educativo:

1. **Lumin**: Un nombre que refleja la iluminación del conocimiento y la sabiduría.
2.


  Intento 2: ¡Aquí te presento algunas propuestas de nombres creativos para un asistente de IA educativo:

1. **Lumin**: una combinación de "lumbrera" y "computadora", sugiriendo la luz del conoc


  Intento 3: Aquí te presento algunas opciones creativas para un nombre de asistente de IA educativo:

1. **Lectrio**: combinación de aprendizaje (lecture) y tecnología (rio).
2. **EduXero


---
## Ejercicio 3: Sistema vs Usuario

**Instrucción:** Crea 3 system prompts diferentes para un mismo tema y compara cómo cambia el 
comportamiento del modelo. Por ejemplo: un tutor estricto, uno amigable y uno sarcástico.

In [4]:
# Escribe aquí tu código para probar system prompts
system_prompts = [
    "Eres un tutor de programación estricto y exigente. Corriges cada error.",
    "Eres un tutor de programación muy paciente y alentador. Usas emojis.",
    "Eres un tutor de programación con humor sarcástico, pero enseñas bien."
]

pregunta = "¿Qué es una variable en Python?"

for sp in system_prompts:
    print(f"\n{'='*50}")
    print(f"System: {sp[:40]}...")
    print('='*50)
    response = client.chat.completions.create(
        model=MODELO_RAPIDO,
        messages=[
            {"role": "system", "content": sp},
            {"role": "user", "content": pregunta}
        ],
        temperature=0.7,
        max_tokens=150
    )
    print(f"Respuesta: {response.choices[0].message.content}")


System: Eres un tutor de programación estricto y...


Respuesta: Una variable en Python es un nombre dado a un valor en memoria, que se puede utilizar para almacenar y acceder a ese valor en un programa. Las variables se utilizan para almacenar y manipular datos en el código.

En Python, las variables no tienen un tipo de dato específico asignado en el momento de su declaración, a diferencia de otros lenguajes de programación como C o Java. En su lugar, el tipo de dato de una variable se determina en el momento en que se asigna un valor a ella.

Por ejemplo, si ejecutas el código siguiente:

```python
mi_variable = 5
```

En este caso, la variable `mi_variable` almacenará el valor entero `5

System: Eres un tutor de programación muy pacien...


Respuesta: ¡Hola! 🌞 Una variable en Python es una contenedora que almacena un valor 📁. Puedes pensar en una variable como un nombre que identifica un valor en específico, de manera que puedas referirte a ese valor con facilidad en tu código.

Por ejemplo, si quieres almacenar el nombre de una persona, puedes crear una variable llamada "nombre" y asignarle el valor "Pedro" 🤔. Luego, puedes usar la variable "nombre" en cualquier parte de tu código para acceder al valor "Pedro".

```python
nombre = "Pedro"
print(nombre)  # imprime "Pedro"
```

En Python, puedes asign

System: Eres un tutor de programación con humor ...


Respuesta: Una variable en Python, o en cualquier lenguaje de programación, es como un contenedor donde puedes almacenar información. Imagina que tienes un bolígrafo y un papel. El bolígrafo es como una variable y el papel es como el espacio donde puedes escribir algo.

En Python, cuando creas una variable, le das un nombre (llamado nombre de variable o identificador) y un valor. El valor puede ser un número, una cadena de texto, una lista, un diccionario, etc. El nombre de la variable es como una etiqueta que identifica qué tipo de información está dentro del contenedor.

Por ejemplo, si creas una variable llamada `edad` y le asign


---
## Ejercicio 4: Streaming en Tiempo Real

**Instrucción:** Implementa una llamada con streaming usando el cliente de Groq directo. 
Muestra cada chunk a medida que llega.

In [5]:
# Escribe aquí tu código para streaming
print("Respuesta en streaming:")
print("-" * 40)

stream = client.chat.completions.create(
    model=MODELO_RAPIDO,
    messages=[{"role": "user", "content": "Explícame qué es el streaming en API en 2 oraciones."}],
    temperature=0.3,
    max_tokens=100,
    stream=True
)

respuesta_completa = ""
for chunk in stream:
    if chunk.choices and chunk.choices[0].delta.content:
        contenido = chunk.choices[0].delta.content
        print(contenido, end="", flush=True)
        respuesta_completa += contenido

print("\n" + "-" * 40)
print(f"\nTotal caracteres: {len(respuesta_completa)}")

Respuesta en streaming:
----------------------------------------


El

 streaming

 en

 API

...

 tipo

 de

 contenido

.


----------------------------------------

Total caracteres: 462


---
## Ejercicio 5: Chatbot con Memoria (Desafío)

**Instrucción:** Implementa un chatbot simple que mantenga el historial de conversación en una lista 
de mensajes. Debe recordar el nombre del usuario y responder preguntas de seguimiento.

**Pistas:**
- Usa una lista `messages` que incluya system, user y assistant roles
- Después de cada respuesta, agregala a la lista
- Para cada nuevo mensaje, envía toda la lista al modelo

In [6]:
# Escribe aquí tu chatbot con memoria
messages = [
    {"role": "system", "content": "Eres un asistente amigable. Recuerdas la conversación."}
]

# Simula una conversación
preguntas = [
    "Hola, me llamo Ana.",
    "¿Cómo me llamo?",
    "¿De qué hablamos recién?"
]

for pregunta in preguntas:
    print(f"\nUsuario: {pregunta}")
    messages.append({"role": "user", "content": pregunta})

    response = client.chat.completions.create(
        model=MODELO_RAPIDO,
        messages=messages,
        temperature=0.3,
        max_tokens=100
    )

    respuesta = response.choices[0].message.content
    print(f"Asistente: {respuesta}")
    messages.append({"role": "assistant", "content": respuesta})

print("\n" + "=" * 40)
print(f"Historial completo: {len(messages)} mensajes")


Usuario: Hola, me llamo Ana.


Asistente: Hola Ana, encantado de conocerte. ¿En qué puedo ayudarte hoy? ¿Quieres hablar sobre algo en particular o simplemente charlar un rato?

Usuario: ¿Cómo me llamo?


Asistente: ¡Claro que sí! Me acabo de enterar de que te llamas Ana. ¿Te gusta ese nombre?

Usuario: ¿De qué hablamos recién?


Asistente: Recién hablamos de que me acabo de enterar de que te llamas Ana. También te pregunté si te gustaba ese nombre. ¿Quieres seguir hablando sobre eso o cambiar de tema?

Historial completo: 7 mensajes


---
## Entregable

Completa todos los ejercicios y documenta:
1. ¿Qué modelo de Groq te gustó más y por qué?
2. ¿Cómo afecta la temperature a las respuestas?
3. ¿Qué ventaja tiene el streaming?
4. ¿Por qué es importante la memoria en un chatbot?

Responde en una celda markdown abajo.

### Tus respuestas:

1. 
2. 
3. 
4. 